# Clase 2.1 y 2.2: Preprocesamiento, Limpieza y Escalado

El 80% de nuestro trabajo como Científicos de Datos es limpiar los datos. Hoy vamos a tratar con valores nulos (El Rompecabezas Dañado) y escalado dimensional (El Gigante y el Enano).

## 1. Tratamiento de Valores Nulos y Codificación

Imagina que tenemos logs de red donde algunos valores de transmisión fallaron, y las alertas vienen en formato texto. Los algoritmos no entienden texto, y fallan si ven valores vacíos (`NaN`).

In [ ]:
import pandas as pd
import numpy as np

data = {
    'request_id': [101, 102, 103, 104, 105],
    'bytes_transmitted': [1500, np.nan, 23000, 1800, 45000],
    'alert_type': ['Malware', 'Phishing', 'Legitimate', 'Malware', np.nan]
}
df = pd.DataFrame(data)
print("Datos Crudos:")
display(df)

### Imputación por la Mediana

Si usamos el promedio (media), el valor `45000` sesgará mucho el resultado. Usaremos la **mediana** (el valor seguro del medio) para rellenar los bytes faltantes.

In [ ]:
mediana_bytes = df['bytes_transmitted'].median()
df['bytes_transmitted'] = df['bytes_transmitted'].fillna(mediana_bytes)
display(df)

### Eliminación y One-Hot Encoding

Si no sabemos la etiqueta (el objetivo), no podemos entrenar. Eliminamos las filas sin `alert_type`. Luego, aplicamos las *Cajas de Luz* (One-Hot Encoding) para no darle peso artificial a las alertas.

In [ ]:
df = df.dropna(subset=['alert_type'])

# Aplicar One-Hot Encoding
df_encoded = pd.get_dummies(df, columns=['alert_type'], prefix='type', dtype=int)
display(df_encoded)

## 2. Escalamiento y Outliers (El Enano y el Gigante)

Un paquete de red pesa 1,000,000 bytes, pero una conexión dura 0.5 segundos. Si calculamos distancias sin escalar, los bytes dominarán el universo.

In [ ]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler

data_network = {'packet_size_kb': [45, 52, 48, 60, 1200, 50, 42, 55]}
df_net = pd.DataFrame(data_network)

Q1 = df_net['packet_size_kb'].quantile(0.25)
Q3 = df_net['packet_size_kb'].quantile(0.75)
IQR = Q3 - Q1

lim_inf = Q1 - 1.5 * IQR
lim_sup = Q3 + 1.5 * IQR

df_clean = df_net[(df_net['packet_size_kb'] >= lim_inf) & (df_net['packet_size_kb'] <= lim_sup)].copy()

scaler_std = StandardScaler()
df_clean['packet_std'] = scaler_std.fit_transform(df_clean[['packet_size_kb']])

display(df_clean)